# SQL Queries for Final Project

This notebook contains the SQL portion of the final project.  
We use SQLite in Python to create tables and run the required queries.

The goal of these queries is to support the main project question: whether aging demand and labor-market pressure in China are consistent with a potential caregiver training and dispatch opportunity.

In [12]:
import pandas as pd
import sqlite3 as sq

In [13]:
# create a connection to the SQLite database
connection = sq.connect("final_project.sqlite")

In [14]:
# Load CSV files
wb_macro = pd.read_csv("data/wb_macro.csv")
country_dim = pd.read_csv("data/country_dim.csv")
china_ops = pd.read_csv("data/china_ops.csv")

In [15]:
# Write the dataframes into SQLite tables
wb_macro.to_sql("wb_macro", connection, if_exists="replace", index=False)
country_dim.to_sql("country_dim", connection, if_exists="replace", index=False)
china_ops.to_sql("china_ops", connection, if_exists="replace", index=False)

6

In [27]:
# Setup row-count check
query = """
SELECT 'wb_macro' AS table_name, COUNT(*) AS row_count FROM wb_macro
UNION ALL
SELECT 'country_dim' AS table_name, COUNT(*) AS row_count FROM country_dim
UNION ALL
SELECT 'china_ops' AS table_name, COUNT(*) AS row_count FROM china_ops;
"""

setup_check_output = pd.read_sql(query, connection)
setup_check_output

,table_name,row_count
0,wb_macro,105
1,country_dim,3
2,china_ops,6


## Query 1. Country-Level Summary Statistics

**What this query does:**  
This query summarizes the main aging, nurse supply, income, and labor-pressure variables by country.

**How it works:**  
It uses `AVG()` and `GROUP BY country` on the `wb_macro` table.

**Why it matters:**  
It provides a simple baseline comparison across China, Japan, and South Korea.

In [16]:
query = """
SELECT
    country,
    ROUND(AVG(pop65_pct), 2) AS avg_pop65_pct,
    ROUND(AVG(nurses_per_1000), 2) AS avg_nurses_per_1000,
    ROUND(AVG(gdp_pc_usd), 2) AS avg_gdp_pc_usd,
    ROUND(AVG(elderly_per_nurse_est), 2) AS avg_elderly_per_nurse_est
FROM wb_macro
GROUP BY country
ORDER BY avg_elderly_per_nurse_est DESC;
"""

query1_output = pd.read_sql(query, connection)
query1_output

,country,avg_pop65_pct,avg_nurses_per_1000,avg_gdp_pc_usd,avg_elderly_per_nurse_est
0,China,8.78,1.71,4744.23,58.25
1,"Korea, Rep.",10.33,4.75,21725.38,24.83
2,Japan,21.55,9.57,37670.55,21.69


**Interpretation:**

This query shows that China has the highest average estimated number of older adults per nurse in the sample (**58.25**), while Japan has the oldest average population profile (**21.55% ages 65+**) and the highest average nurse density (**9.57 per 1,000**). South Korea falls between China and Japan on most measures.

This matters for the project because it suggests that China combines meaningful aging demand with relatively tighter labor conditions in our proxy measure. For the target audience, that supports the idea that a caregiver training and dispatch business may be more relevant in China than in the benchmark countries.

## Query 2. China vs. Benchmark Group Comparison

**What this query does:**  
This query compares China with the benchmark group on average aging, nurse supply, income, and labor pressure.

**How it works:**  
It joins `wb_macro` with `country_dim`, then groups by `region_group` and calculates averages.

**Why it matters:**  
It shows whether China looks systematically different from the benchmark countries as a group.

In [17]:
query = """
SELECT
    c.region_group,
    ROUND(AVG(w.pop65_pct), 2) AS avg_pop65_pct,
    ROUND(AVG(w.nurses_per_1000), 2) AS avg_nurses_per_1000,
    ROUND(AVG(w.gdp_pc_usd), 2) AS avg_gdp_pc_usd,
    ROUND(AVG(w.elderly_per_nurse_est), 2) AS avg_elderly_per_nurse_est
FROM wb_macro AS w
LEFT JOIN country_dim AS c
    ON w.iso3 = c.iso3
GROUP BY c.region_group
ORDER BY avg_elderly_per_nurse_est DESC;
"""

query2_output = pd.read_sql(query, connection)
query2_output

,region_group,avg_pop65_pct,avg_nurses_per_1000,avg_gdp_pc_usd,avg_elderly_per_nurse_est
0,China,8.78,1.71,4744.23,58.25
1,Benchmark,15.94,6.53,29697.97,23.67


**Interpretation:**

This query shows that China has a much higher average estimated number of older adults per nurse (**58.25**) than the benchmark group (**23.67**). The benchmark group also has much higher average nurse density (**6.53 vs. 1.71**) and a much higher average income level.

This matters for the project because it suggests that China looks systematically tighter on the labor side of elder care than the nearby benchmark countries as a group. For the target audience, that strengthens the case for investigating services that help recruit, train, and place care workers more efficiently.

## Query 3. China Macro and Operations View by Year

**What this query does:**  
This query combines China’s macro indicators with China’s operations data by year.

**How it works:**  
It joins `wb_macro` and `china_ops` on `year` and keeps only China for 2019–2024.

**Why it matters:**  
It connects the broader macro story to more business-relevant elder-care capacity data in China.

In [19]:
query = """
SELECT
    w.year,
    ROUND(w.pop65_pct, 2) AS pop65_pct,
    ROUND(w.nurses_per_1000, 2) AS nurses_per_1000,
    ROUND(w.elderly_per_nurse_est, 2) AS elderly_per_nurse_est,
    ROUND(c.age65_total_mn, 2) AS age65_total_mn,
    ROUND(c.eldercare_beds_mn, 2) AS eldercare_beds_mn,
    ROUND(c.registered_nurses_mn, 2) AS registered_nurses_mn,
    ROUND(c.beds_per_1000_65plus, 2) AS beds_per_1000_65plus
FROM wb_macro AS w
LEFT JOIN china_ops AS c
    ON w.year = c.year
WHERE w.iso3 = 'CHN'
  AND w.year BETWEEN 2019 AND 2024
ORDER BY w.year;
"""

query3_output = pd.read_sql(query, connection)
query3_output

,year,pop65_pct,nurses_per_1000,elderly_per_nurse_est,age65_total_mn,eldercare_beds_mn,registered_nurses_mn,beds_per_1000_65plus
0,2019,12.06,3.12,38.62,169.79,7.61,4.43,44.84
1,2020,12.65,3.30,38.30,178.44,8.24,4.71,46.17
2,2021,13.21,3.52,37.53,186.51,8.13,5.02,43.62
3,2022,13.78,3.67,37.59,194.62,8.22,5.20,42.25
4,2023,14.32,NaN,NaN,201.96,8.20,5.63,40.61
5,2024,14.67,NaN,NaN,206.63,7.99,5.84,38.67


**Interpretation:**

This query shows that China’s 65+ population share rose steadily from **12.06%** in 2019 to **14.67%** in 2024, while the total 65+ population also continued to increase. Over the same period, elder-care beds per 1,000 older adults declined from **44.84** to **38.67**, even though registered nurses increased from **4.43 million** to **5.84 million**. The workforce pressure proxy is available through 2022 and remains relatively high in those observed years.

This matters for the project because it suggests that demand growth may be outpacing capacity growth on a per-older-adult basis. For the target audience, that makes the labor side of elder care especially important and supports further investigation of a caregiver training and dispatch business.

## Query 4. Highest Labor-Pressure Year by Country

**What this query does:**  
This query finds the highest observed labor-pressure year for each country.

**How it works:**  
It uses `ROW_NUMBER()` as a window function to rank years within each country by `elderly_per_nurse_est`.

**Why it matters:**  
It shows when labor pressure was most severe in each market.

In [20]:
query = """
SELECT
    country,
    year,
    elderly_per_nurse_est,
    pressure_rank
FROM (
    SELECT
        country,
        year,
        ROUND(elderly_per_nurse_est, 2) AS elderly_per_nurse_est,
        ROW_NUMBER() OVER (
            PARTITION BY country
            ORDER BY elderly_per_nurse_est DESC
        ) AS pressure_rank
    FROM wb_macro
    WHERE elderly_per_nurse_est IS NOT NULL
)
WHERE pressure_rank = 1
ORDER BY elderly_per_nurse_est DESC;
"""

query4_output = pd.read_sql(query, connection)
query4_output

,country,year,elderly_per_nurse_est,pressure_rank
0,China,2003,77.50,1
1,"Korea, Rep.",1994,50.88,1
2,Japan,2016,23.78,1


**Interpretation:**

This query shows that China had the highest peak value of the labor-pressure proxy in the sample, reaching **77.50** older adults per nurse in **2003**. Korea’s highest observed value was **50.88** in **1994**, and Japan’s was **23.78** in **2016**.

This matters for the project because it suggests that China experienced the most severe observed labor-pressure conditions among the three countries in our sample. For a caregiver training and dispatch business, that is consistent with the idea that workforce constraints may be especially relevant in China.

## Query 5. Year-over-Year Change in China’s 65+ Population Share

**What this query does:**  
This query calculates the year-over-year change in China’s 65+ population share.

**How it works:**  
It uses the `LAG()` window function on `pop65_pct`, ordered by year.

**Why it matters:**  
It helps show whether aging demand is still growing over time in China.

In [21]:
query = """
SELECT
    year,
    ROUND(pop65_pct, 2) AS pop65_pct,
    ROUND(
        pop65_pct - LAG(pop65_pct) OVER (
            ORDER BY year
        ),
        2
    ) AS yoy_change_pop65_pct
FROM wb_macro
WHERE iso3 = 'CHN'
  AND year BETWEEN 2019 AND 2024
ORDER BY year;
"""

query5_output = pd.read_sql(query, connection)
query5_output

,year,pop65_pct,yoy_change_pop65_pct
0,2019,12.06,NaN
1,2020,12.65,0.58
2,2021,13.21,0.56
3,2022,13.78,0.58
4,2023,14.32,0.53
5,2024,14.67,0.35


**Interpretation:**

This query shows that China’s 65+ population share increased in every observed year after 2019. The year-over-year increases are modest, but they remain positive throughout 2020–2024.

This matters for the project because it supports the idea that elder-care demand is continuing to grow rather than flattening out. For the target audience, that makes the market more relevant as a long-term opportunity rather than a short-term trend.

## Query 6. Years with Above-Average Labor Pressure Within Each Country

**What this query does:**  
This query identifies the years when each country’s labor-pressure proxy was above its own average level.

**How it works:**  
It uses a correlated subquery to compare each row’s `elderly_per_nurse_est` with that country’s average.

**Why it matters:**  
It helps distinguish normal periods from relatively tight labor-market periods within each country.

In [22]:
query = """
SELECT
    w.country,
    w.year,
    ROUND(w.elderly_per_nurse_est, 2) AS elderly_per_nurse_est
FROM wb_macro AS w
WHERE w.elderly_per_nurse_est IS NOT NULL
  AND w.elderly_per_nurse_est > (
      SELECT AVG(w2.elderly_per_nurse_est)
      FROM wb_macro AS w2
      WHERE w2.country = w.country
        AND w2.elderly_per_nurse_est IS NOT NULL
  )
ORDER BY w.country, w.year;
"""

query6_output = pd.read_sql(query, connection)
query6_output

,country,year,elderly_per_nurse_est
0,China,1990,63.26
1,China,1995,65.58
2,China,1996,65.87
3,China,1997,66.33
4,China,1998,67.79
5,China,1999,68.97
6,China,2000,70.32
7,China,2001,71.70
8,China,2002,76.38
9,China,2003,77.50


**Interpretation:**

This query identifies the years in which each country’s labor-pressure proxy was above that country’s own long-run average. In the output, China shows a long stretch of above-average pressure years, especially from the mid-1990s through the 2000s. Japan’s above-average years are more concentrated in the later part of the sample, while Korea’s visible above-average years appear earlier in the sample.

This matters for the project because it helps distinguish ordinary conditions from relatively tight labor-market periods within each country. For the target audience, the result is useful because it suggests that China experienced a more extended period of elevated labor pressure, which is consistent with a market where caregiver training and dispatch services may be especially relevant.

## Query 7. Years When China’s Bed Capacity per Older Adult Fell Below Its Period Average

**What this query does:**  
This query identifies the years when China’s bed capacity per older adult was below its period average.

**How it works:**  
It uses a subquery to calculate the average `beds_per_1000_65plus` and filters years below that value.

**Why it matters:**  
It helps show when elder-care capacity may have lagged behind the growth of the older population.

In [23]:
query = """
SELECT
    year,
    ROUND(beds_per_1000_65plus, 2) AS beds_per_1000_65plus
FROM china_ops
WHERE beds_per_1000_65plus < (
    SELECT AVG(beds_per_1000_65plus)
    FROM china_ops
    WHERE beds_per_1000_65plus IS NOT NULL
)
ORDER BY year;
"""

query7_output = pd.read_sql(query, connection)
query7_output

,year,beds_per_1000_65plus
0,2022,42.25
1,2023,40.61
2,2024,38.67


**Interpretation:**

This query shows that China’s elder-care bed capacity fell below its 2019–2024 period average in **2022, 2023, and 2024**. The values continue to decline across those years, from **42.25** beds per 1,000 older adults in 2022 to **38.67** in 2024.

This matters for the project because it suggests that elder-care capacity did not keep pace with the growth of the older population in the later part of the period. For the target audience, that strengthens the case that labor and service delivery constraints may become more important as aging demand continues to rise.

## Query 8. Most Recent Non-Missing Labor-Pressure Observation by Country

**What this query does:**  
This query finds the most recent year with a non-missing labor-pressure value for each country.

**How it works:**  
It uses `ROW_NUMBER()` as a window function, ordered by year descending within each country.

**Why it matters:**  
It provides the latest directly comparable labor-pressure snapshot across the three countries.

In [24]:
query = """
SELECT
    country,
    year,
    ROUND(elderly_per_nurse_est, 2) AS elderly_per_nurse_est
FROM (
    SELECT
        country,
        year,
        elderly_per_nurse_est,
        ROW_NUMBER() OVER (
            PARTITION BY country
            ORDER BY year DESC
        ) AS recent_rank
    FROM wb_macro
    WHERE elderly_per_nurse_est IS NOT NULL
)
WHERE recent_rank = 1
ORDER BY country;
"""

query8_output = pd.read_sql(query, connection)
query8_output

,country,year,elderly_per_nurse_est
0,China,2022,37.59
1,Japan,2022,22.65
2,"Korea, Rep.",2022,19.24


**Interpretation:**

This query shows the most recent year with a non-missing labor-pressure proxy for each country. In the current data, the most recent non-missing observation is **2022** for all three countries: **37.59** for China, **22.65** for Japan, and **19.24** for Korea, Rep.

This matters for the project because it provides the latest directly comparable snapshot of labor pressure across the three markets. For the target audience, the result suggests that China still appears to face tighter elder-care labor conditions than the benchmark countries in the most recent comparable year.

## Query 9. China and Benchmark Group Comparison in the Most Recent Comparable Year

**What this query does:**  
This query compares China and the benchmark group in the most recent year with comparable labor-pressure data.

**How it works:**  
It joins `wb_macro` with `country_dim`, filters to 2022, and groups by `region_group`.

**Why it matters:**  
It shows whether China still looks tighter on the labor side in the latest comparable year.

In [25]:
query = """
SELECT
    c.region_group,
    ROUND(AVG(w.elderly_per_nurse_est), 2) AS avg_elderly_per_nurse_est_2022,
    ROUND(AVG(w.nurses_per_1000), 2) AS avg_nurses_per_1000_2022,
    ROUND(AVG(w.pop65_pct), 2) AS avg_pop65_pct_2022
FROM wb_macro AS w
LEFT JOIN country_dim AS c
    ON w.iso3 = c.iso3
WHERE w.year = 2022
  AND w.elderly_per_nurse_est IS NOT NULL
GROUP BY c.region_group
ORDER BY avg_elderly_per_nurse_est_2022 DESC;
"""

query9_output = pd.read_sql(query, connection)
query9_output

,region_group,avg_elderly_per_nurse_est_2022,avg_nurses_per_1000_2022,avg_pop65_pct_2022
0,China,37.59,3.67,13.78
1,Benchmark,20.95,11.03,23.44


**Interpretation:**

This query compares China with the benchmark group in the most recent year where the labor-pressure proxy is available for all countries, which is **2022**. In that year, China had a much higher average estimated number of older adults per nurse (**37.59**) than the benchmark group (**20.95**), while the benchmark group had much higher nurse density (**11.03 vs. 3.67**) and a higher 65+ population share (**23.44% vs. 13.78%**).

This matters for the project because it shows that even in the most recent comparable year, China still appears tighter on the labor side of elder care than the benchmark group. For the target audience, that supports the idea that workforce-related elder-care services may be especially relevant in China.

## Query 10. Years When China’s Labor Pressure Exceeded Both Japan and Korea

**What this query does:**  
This query identifies the years when China’s labor-pressure proxy was higher than both Japan’s and Korea’s.

**How it works:**  
It self-joins `wb_macro` three times by year for China, Japan, and Korea, then filters for years when China is higher than both.

**Why it matters:**  
It shows whether China’s labor-pressure conditions were consistently tighter than both benchmark countries.

In [26]:
query = """
SELECT
    chn.year,
    ROUND(chn.elderly_per_nurse_est, 2) AS china_elderly_per_nurse_est,
    ROUND(jpn.elderly_per_nurse_est, 2) AS japan_elderly_per_nurse_est,
    ROUND(kor.elderly_per_nurse_est, 2) AS korea_elderly_per_nurse_est,
    ROUND(chn.elderly_per_nurse_est - jpn.elderly_per_nurse_est, 2) AS china_minus_japan,
    ROUND(chn.elderly_per_nurse_est - kor.elderly_per_nurse_est, 2) AS china_minus_korea
FROM wb_macro AS chn
LEFT JOIN wb_macro AS jpn
    ON chn.year = jpn.year
   AND jpn.iso3 = 'JPN'
LEFT JOIN wb_macro AS kor
    ON chn.year = kor.year
   AND kor.iso3 = 'KOR'
WHERE chn.iso3 = 'CHN'
  AND chn.elderly_per_nurse_est IS NOT NULL
  AND jpn.elderly_per_nurse_est IS NOT NULL
  AND kor.elderly_per_nurse_est IS NOT NULL
  AND chn.elderly_per_nurse_est > jpn.elderly_per_nurse_est
  AND chn.elderly_per_nurse_est > kor.elderly_per_nurse_est
ORDER BY chn.year;
"""

query10_output = pd.read_sql(query, connection)
query10_output

,year,china_elderly_per_nurse_est,japan_elderly_per_nurse_est,korea_elderly_per_nurse_est,china_minus_japan,china_minus_korea
0,1996,65.87,19.56,49.91,46.31,15.96
1,1998,67.79,19.86,23.29,47.93,44.50
2,2000,70.32,20.15,23.64,50.16,46.68
3,2002,76.38,21.63,21.48,54.76,54.91
4,2004,76.99,21.90,22.29,55.09,54.69
5,2006,74.43,22.53,23.30,51.91,51.13
6,2008,66.42,22.78,23.22,43.64,43.20
7,2010,57.17,22.43,23.33,34.75,33.84
8,2012,49.86,22.84,23.85,27.01,26.01
9,2014,44.67,23.46,22.30,21.21,22.37


**Interpretation:**

This query shows the years in which China’s labor-pressure proxy exceeded both Japan and Korea at the same time. In the returned output, China is higher than both benchmark countries in every comparable year shown, from **1996** through **2022**.

The gap is especially large in the early and middle part of the sample, and although it narrows over time, China still remains above both countries in the most recent comparable year. For example, in **2022**, China’s value is **37.59**, compared with **22.65** for Japan and **19.24** for Korea.

This matters for the project because it reinforces the broader finding that China appears consistently tighter on the labor side of elder care than the benchmark countries whenever comparable observations are available. For the target audience, that strengthens the case for exploring caregiver training and dispatch as a potentially relevant business opportunity in China.